# Distance Metric Comparison (Appendix A, Figure 6)

Reproduces **Figure 6** from Appendix A of the DTR paper.

This notebook compares three distance metrics for computing DTR-like scores:
- **JSD** (Jensen-Shannon Divergence): symmetric, bounded, default choice for DTR
- **KLD** (Kullback-Leibler Divergence): asymmetric, unbounded
- **Cosine Distance**: operates on raw hidden states (no vocab projection)

Key finding: JSD achieves the strongest and most consistent correlation with accuracy,
followed by KLD, with cosine distance being the weakest.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

from dtr.metrics.dtr import compute_dtr
from dtr.metrics.distances import jsd, kld, cosine_distance
from dtr.analysis.sensitivity import recompute_dtr_from_jsd
from dtr.analysis.correlation import compute_binned_correlation
from dtr.utils.io import load_json

## Configuration

In [ ]:
MODEL = "deepseek-r1"
BENCHMARKS = ["aime_2025", "hmmt_2025"]  # Figure 6 focuses on these two
BENCHMARK_LABELS = ["AIME 2025", "HMMT 2025"]

METRICS_DIR = Path("..") / "results" / "metrics"
RAW_DIR = Path("..") / "results" / "raw"

DISTANCE_METRICS = ["JSD", "KLD", "Cosine"]
DISTANCE_KEYS = ["dtr_jsd", "dtr_kld", "dtr_cosine"]
DISTANCE_COLORS = [sns.color_palette()[0], sns.color_palette()[1], sns.color_palette()[2]]

## Load Per-Sample Metrics

Load pre-computed DTR values using each distance metric.

In [ ]:
all_data = {}

for bench in BENCHMARKS:
    metrics_path = METRICS_DIR / MODEL / bench / "per_sample_metrics.json"
    if metrics_path.exists():
        data = load_json(str(metrics_path))
        all_data[bench] = data
        print(f"Loaded {bench}: {len(data)} samples")
        
        # Check which distance metrics are available
        sample = data[0]
        available = [k for k in DISTANCE_KEYS if k in sample]
        print(f"  Available distance keys: {available}")
        print(f"  All keys: {list(sample.keys())}")
    else:
        print(f"WARNING: {metrics_path} not found")

## Compute Correlations for Each Distance Metric

For each benchmark and distance metric, compute the binned Pearson r with accuracy.

In [ ]:
corr_results = {}

for bench in BENCHMARKS:
    if bench not in all_data:
        continue
    
    data = all_data[bench]
    accuracies = np.array([d["correct"] for d in data], dtype=float)
    
    bench_corrs = {}
    for metric_name, metric_key in zip(DISTANCE_METRICS, DISTANCE_KEYS):
        # Try the specific key, fall back to generic "dtr" for JSD
        if metric_key == "dtr_jsd" and metric_key not in data[0]:
            metric_key = "dtr"  # default DTR uses JSD
        
        if all(metric_key in d for d in data):
            values = np.array([d[metric_key] for d in data])
            corr = compute_binned_correlation(values, accuracies)
            bench_corrs[metric_name] = corr
            print(f"{bench} / {metric_name}: r = {corr:.3f}")
        else:
            bench_corrs[metric_name] = np.nan
            print(f"{bench} / {metric_name}: not available")
    
    corr_results[bench] = bench_corrs

## Figure 6: Side-by-Side Correlation Comparison

Scatter plots comparing DTR (computed with each distance metric) vs accuracy
for AIME 2025 and HMMT 2025.

In [ ]:
NUM_BINS = 10

fig, axes = plt.subplots(len(BENCHMARKS), len(DISTANCE_METRICS), figsize=(15, 5 * len(BENCHMARKS)))
if len(BENCHMARKS) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle(
    "Figure 6: Distance Metric Comparison for DTR",
    fontsize=16, fontweight="bold", y=1.02,
)

for row, (bench, bench_label) in enumerate(zip(BENCHMARKS, BENCHMARK_LABELS)):
    if bench not in all_data:
        for col in range(len(DISTANCE_METRICS)):
            axes[row, col].text(0.5, 0.5, "No data", ha="center", va="center",
                                 transform=axes[row, col].transAxes)
        continue
    
    data = all_data[bench]
    accuracies = np.array([d["correct"] for d in data], dtype=float)
    
    for col, (metric_name, metric_key, color) in enumerate(
        zip(DISTANCE_METRICS, DISTANCE_KEYS, DISTANCE_COLORS)
    ):
        ax = axes[row, col]
        
        # Resolve key
        actual_key = metric_key
        if metric_key == "dtr_jsd" and metric_key not in data[0]:
            actual_key = "dtr"
        
        if not all(actual_key in d for d in data):
            ax.text(0.5, 0.5, "Not computed", ha="center", va="center",
                     transform=ax.transAxes, fontsize=12)
            ax.set_title(f"{metric_name} ({bench_label})", fontsize=11)
            continue
        
        values = np.array([d[actual_key] for d in data])
        
        # Bin and plot
        bin_edges = np.percentile(values, np.linspace(0, 100, NUM_BINS + 1))
        bin_edges = np.unique(bin_edges)
        bin_centers, bin_means, bin_stds = [], [], []
        
        for i in range(len(bin_edges) - 1):
            mask = (values >= bin_edges[i])
            if i < len(bin_edges) - 2:
                mask &= (values < bin_edges[i + 1])
            else:
                mask &= (values <= bin_edges[i + 1])
            if mask.sum() > 0:
                bin_centers.append(values[mask].mean())
                bin_means.append(accuracies[mask].mean())
                bin_stds.append(accuracies[mask].std() / np.sqrt(mask.sum()))
        
        # Scatter points
        ax.scatter(values, accuracies, alpha=0.1, s=8, color=color, rasterized=True)
        
        # Binned means
        ax.errorbar(
            bin_centers, bin_means, yerr=bin_stds,
            fmt="o-", color=color, markersize=6, linewidth=2, capsize=3,
        )
        
        # Correlation annotation
        r = corr_results.get(bench, {}).get(metric_name, np.nan)
        ax.text(
            0.05, 0.95, f"r = {r:.3f}",
            transform=ax.transAxes, fontsize=12, fontweight="bold",
            verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
        )
        
        ax.set_title(f"{metric_name} ({bench_label})", fontsize=11)
        ax.set_xlabel(f"DTR ({metric_name})", fontsize=10)
        if col == 0:
            ax.set_ylabel("Accuracy", fontsize=11)

plt.tight_layout()
plt.savefig("../results/figures/fig6_distance_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Summary Bar Chart: Correlation by Distance Metric

Direct comparison of correlation strength for each distance metric across benchmarks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(BENCHMARKS))
width = 0.25

for i, (metric_name, color) in enumerate(zip(DISTANCE_METRICS, DISTANCE_COLORS)):
    corrs = [
        corr_results.get(bench, {}).get(metric_name, 0)
        for bench in BENCHMARKS
    ]
    bars = ax.bar(
        x + i * width, corrs,
        width=width, color=color, edgecolor="white",
        label=metric_name,
    )
    
    for bar, val in zip(bars, corrs):
        if not np.isnan(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=9, fontweight="bold",
            )

ax.set_xticks(x + width)
ax.set_xticklabels(BENCHMARK_LABELS, fontsize=11)
ax.set_ylabel("Pearson r (DTR vs Accuracy)", fontsize=12)
ax.set_title("Distance Metric Comparison: Correlation Strength", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Key Takeaway

JSD is the best distance metric for DTR computation:
- **JSD >> KLD >> Cosine** in terms of correlation strength with accuracy
- JSD's symmetry and bounded range [0, ln(2)] make it numerically stable
- Cosine distance on raw hidden states is insufficient because it doesn't capture
  the distributional differences that matter for predicting answer quality